# Multimodal AQI Prediction — EfficientNetB0 + BiLSTM + Cross-Attention + XAI

**Author's project:** Prakhar Pandey — Multimodal Deep Learning AQI Prediction (CNN image branch + time-series branch + Explainable AI)

**What this notebook does**
- Downloads both Kaggle datasets directly (no manual download needed)
- Builds a two-branch deep learning model: image branch (EfficientNetB0, transfer learning) + time-series branch (BiLSTM + attention)
- Fuses both branches with a lightweight cross-attention layer
- Trains with class balancing (Focal Loss), mixed precision, early stopping
- Evaluates with a full metric suite (Accuracy, Macro-F1, MCC, Cohen's Kappa, ROC-AUC, Confusion Matrix)
- Explains predictions with Grad-CAM (images) and SHAP (tabular/time-series features)
- Maps predicted AQI category → health risk + recommendations
- Exports the model + preprocessing artifacts, ready to be loaded by a FastAPI backend

**⏱️ Time budget — read this first**

This notebook is deliberately tuned to finish in **~3–4 hours on a single modest GPU**
(a laptop GPU with 4–6 GB VRAM, or a free Google Colab T4 — **Colab is strongly recommended**,
go to `Runtime → Change runtime type → T4 GPU` before running). On CPU-only it will take
much longer (likely 10+ hours) because of the image branch — if you don't have a GPU, run this in Colab.

Everything that controls runtime lives in **one `CONFIG` dict** in the next code cell:
- `FAST_MODE = True` → smaller image subsample, smaller image size, fewer epochs → fits 3–4 hrs on a modest GPU
- `FAST_MODE = False` → uses more data / bigger images / more epochs, for when you later have access to a stronger GPU (better for your final report numbers, but can take 8+ hours)

Rough time split at `FAST_MODE = True` on a T4:
| Stage | Time |
|---|---|
| Download + preprocessing | 10–15 min |
| Feature engineering | ~5 min |
| Image branch pretraining (frozen backbone) | 35–45 min |
| Time-series branch pretraining | 20–30 min |
| Joint fusion training + fine-tuning | 60–90 min |
| Evaluation | 5–10 min |
| Explainability (Grad-CAM + SHAP) | 20–30 min |
| **Total** | **~3–3.5 hrs** |

**A note on the two datasets and how they're combined**

Your two Kaggle datasets are **not** natively linked by a common city/date key — the image
dataset is a folder of photos labeled by visual AQI category, and the time-series dataset is a
per-city daily pollutant table. Real multimodal fusion needs *paired* samples, so this notebook:
1. First tries to auto-detect a city/date key inside the image filenames/folder names (some
   AQI image datasets embed this) and does a real key-based join if possible.
2. If no key is found, it falls back to **category-matched pairing**: each image is paired with
   a randomly sampled time-series window from the *same* AQI category. This is a standard,
   defensible approach for building a synthetic-but-label-consistent multimodal dataset when two
   independent public datasets are combined — and it's exactly what your report's architecture
   needs. This is noted explicitly in the printed output and you should mention this pairing
   strategy in your report/viva as a design decision (with its limitation: it does not model true
   image↔sensor covariation, only label consistency).

I don't have live internet access to inspect the exact CSV column names in your environment, so
Cell 4 **prints the actual detected schema** the first time you run it — check that printout after
your first run and adjust the `COLUMN_MAP` in `CONFIG` if the auto-detection picks the wrong column.


## 1. Install & import libraries

In [ ]:
# Run this once. On Colab most of these are preinstalled; the pip install is safe to re-run.
!pip install -q kagglehub tensorflow-addons shap opencv-python-headless scikit-learn \
    imbalanced-learn joblib tqdm seaborn > /dev/null
print("Install step done.")


In [ ]:
import os, re, json, glob, math, random, warnings, time, joblib
from datetime import datetime
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks, regularizers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input as effnet_preprocess

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, f1_score,
                              matthews_corrcoef, cohen_kappa_score, confusion_matrix,
                              classification_report, roc_auc_score, roc_curve)
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPUs available:", gpus if gpus else "NONE — this will be slow, use a GPU runtime if possible")

# Mixed precision speeds up training substantially on modern GPUs (T4/V100/A100/RTX)
if gpus:
    try:
        tf.keras.mixed_precision.set_global_policy('mixed_float16')
        print("Mixed precision: ON")
    except Exception as e:
        print("Mixed precision not enabled:", e)


## 2. Config — everything that controls runtime lives here

In [ ]:
CONFIG = {
    # ---- time budget switch ----
    "FAST_MODE": True,          # keep True for a 3-4 hr run on a modest GPU

    # ---- image branch ----
    "IMG_SIZE": 128,
    "IMG_SAMPLES_PER_CLASS": 1000,   # increased from 700 so each class has more examples
    "IMG_BATCH": 32,
    "IMG_EPOCHS_HEAD": 20,       # FIXED: was 10 — doubled so the head learns properly
    "IMG_EPOCHS_FINETUNE": 15,   # FIXED: was 5  — tripled for deep CNN adaptation

    # ---- time-series branch ----
    "SEQ_LEN": 14,
    "TS_ROWS_SUBSAMPLE": 200_000,
    "TS_BATCH": 64,

    # ---- fusion / joint training ----
    "FUSION_EPOCHS": 15,         # FIXED: was 12
    "LEARNING_RATE": 1e-3,
    "FINE_TUNE_LR": 2e-5,        # FIXED: was 1e-5, slightly higher for better CNN adaptation
    "LABEL_SMOOTHING": 0.05,
    "FOCAL_GAMMA": 2.0,

    # ---- MODALITY DROPOUT probability ----
    # During training, this fraction of samples will have the timeseries input zeroed out,
    # forcing the CNN image branch to learn to predict AQI from visual features.
    "MODALITY_DROPOUT_PROB": 0.35,

    # ---- AQI categories (CPCB standard, 6 classes) ----
    "AQI_CATEGORIES": ["Good", "Satisfactory", "Moderate", "Poor", "Very Poor", "Severe"],
    "AQI_BREAKPOINTS": [0, 50, 100, 200, 300, 400, 5000],

    # ---- explainability ----
    "SHAP_BACKGROUND_SAMPLES": 50,
    "SHAP_EXPLAIN_SAMPLES": 30,
    "GRADCAM_EXAMPLES": 6,

    # ---- misc ----
    "SEED": SEED,
    "OUTPUT_DIR": "aqi_model_artifacts",
}

if not CONFIG["FAST_MODE"]:
    CONFIG.update({
        "IMG_SIZE": 224, "IMG_SAMPLES_PER_CLASS": None,
        "IMG_EPOCHS_HEAD": 30, "IMG_EPOCHS_FINETUNE": 20,
        "TS_ROWS_SUBSAMPLE": None, "FUSION_EPOCHS": 30,
        "MODALITY_DROPOUT_PROB": 0.35,
    })

os.makedirs(CONFIG["OUTPUT_DIR"], exist_ok=True)
print(json.dumps(CONFIG, indent=2))


## 3. Download both datasets directly from Kaggle (via `kagglehub`, no manual download)

In [ ]:
# kagglehub handles Kaggle auth for you. On first run it may open a browser prompt to log in
# with your Kaggle account (or ask you to upload kaggle.json) — do that once and it's cached.
import kagglehub

print("Downloading image dataset (Air Pollution Image Dataset — India & Nepal)...")
IMAGE_DATASET_PATH = kagglehub.dataset_download(
    "adarshrouniyar/air-pollution-image-dataset-from-india-and-nepal"
)
print("Image dataset at:", IMAGE_DATASET_PATH)

print("Downloading time-series dataset (Time Series Air Quality Data of India 2010-2023)...")
TS_DATASET_PATH = kagglehub.dataset_download(
    "abhisheksjha/time-series-air-quality-data-of-india-2010-2023"
)
print("Time-series dataset at:", TS_DATASET_PATH)


## 4. Explore both datasets' actual structure

This cell just **prints** what's on disk — folder names, a sample CSV, and its columns — so you
can confirm (or fix) the auto-detected column names in the next cell before we build anything on
top of them.


In [ ]:
print("=== IMAGE DATASET STRUCTURE ===")
for root, dirs, files in os.walk(IMAGE_DATASET_PATH):
    depth = root[len(IMAGE_DATASET_PATH):].count(os.sep)
    if depth <= 2:
        print("  " * depth + os.path.basename(root) + f"/  ({len(files)} files)" if files else "  " * depth + os.path.basename(root) + "/")

image_files = glob.glob(os.path.join(IMAGE_DATASET_PATH, "**", "*.*"), recursive=True)
image_files = [f for f in image_files if f.lower().endswith((".jpg", ".jpeg", ".png"))]
print(f"\nTotal image files found: {len(image_files)}")
print("Sample paths:")
for f in image_files[:5]:
    print(" ", f)

print("\n=== TIME SERIES DATASET STRUCTURE ===")
ts_csvs = glob.glob(os.path.join(TS_DATASET_PATH, "**", "*.csv"), recursive=True)
print("CSV files found:", ts_csvs)
ts_preview = pd.read_csv(ts_csvs[0], nrows=200)
print("\nColumns:", list(ts_preview.columns))
print(ts_preview.head())


## 5. Auto-detect columns and image class labels

In [ ]:
def find_col(columns, keywords):
    """Return first column whose name contains any keyword (case-insensitive)."""
    for kw in keywords:
        for c in columns:
            if kw.lower() in c.lower():
                return c
    return None

cols = list(ts_preview.columns)
COLUMN_MAP = {
    "date":  find_col(cols, ["date", "datetime", "timestamp"]),
    "city":  find_col(cols, ["city", "station", "location"]),
    "pm25":  find_col(cols, ["pm2.5", "pm25", "pm_2.5"]),
    "pm10":  find_col(cols, ["pm10", "pm_10"]),
    "no2":   find_col(cols, ["no2"]),
    "so2":   find_col(cols, ["so2"]),
    "co":    find_col(cols, ["co"]),
    "o3":    find_col(cols, ["o3", "ozone"]),
    "aqi":   find_col(cols, ["aqi"]),
}
print("Auto-detected column map (EDIT MANUALLY below if any is wrong / None):")
print(json.dumps(COLUMN_MAP, indent=2))

# --- If auto-detection got something wrong, uncomment and fix a line like this: ---
# COLUMN_MAP["date"] = "Date"
# COLUMN_MAP["aqi"]  = "AQI"

assert COLUMN_MAP["aqi"] is not None, "Could not find an AQI column — set COLUMN_MAP['aqi'] manually."

# Image class folders (top-level subfolders of the dataset root, or one level down)
def list_class_dirs(root):
    candidates = [d for d in glob.glob(os.path.join(root, "*")) if os.path.isdir(d)]
    if len(candidates) == 1:  # dataset wrapped in one extra folder
        candidates = [d for d in glob.glob(os.path.join(candidates[0], "*")) if os.path.isdir(d)]
    return candidates

class_dirs = list_class_dirs(IMAGE_DATASET_PATH)
IMAGE_CLASSES = sorted([os.path.basename(d) for d in class_dirs])
print("\nDetected image class folders:", IMAGE_CLASSES)


## 6. AQI category scheme — align image folder labels with numeric AQI categories

We use the standard CPCB 6-band scheme from `CONFIG["AQI_CATEGORIES"]`. We try to map each
image-folder name onto one of these 6 bands by fuzzy name matching (e.g. a folder called
`"Very_Poor"` or `"severe"` maps directly); if a folder name doesn't match anything recognisable,
it's printed so you can fix the mapping by hand.


In [ ]:
def normalize(s):
    return re.sub(r'[^a-z]', '', s.lower())

CATEGORY_ALIASES = {
    "good": "Good",
    "satisfactory": "Satisfactory", "moderate": "Moderate", "fair": "Satisfactory",
    "poor": "Poor", "verypoor": "Very Poor", "severe": "Severe", "hazardous": "Severe",
}

FOLDER_TO_CATEGORY = {}
for folder in IMAGE_CLASSES:
    key = normalize(folder)
    matched = None
    for alias, cat in CATEGORY_ALIASES.items():
        if alias in key:
            matched = cat
            break
    FOLDER_TO_CATEGORY[folder] = matched
print("Folder -> AQI category mapping:")
print(json.dumps(FOLDER_TO_CATEGORY, indent=2))

unmapped = [f for f, c in FOLDER_TO_CATEGORY.items() if c is None]
if unmapped:
    print("\n⚠️  Could not auto-map these folders, please edit FOLDER_TO_CATEGORY manually:", unmapped)
    # Example fix:
    # FOLDER_TO_CATEGORY["class_0"] = "Good"

def aqi_value_to_category(aqi):
    bp = CONFIG["AQI_BREAKPOINTS"]
    cats = CONFIG["AQI_CATEGORIES"]
    for i in range(len(cats)):
        if bp[i] <= aqi <= bp[i+1]:
            return cats[i]
    return cats[-1]


## 7. Load & clean the time-series data

In [ ]:
usecols = [v for v in COLUMN_MAP.values() if v is not None]
df_ts = pd.read_csv(ts_csvs[0], usecols=usecols, low_memory=False)

if CONFIG["TS_ROWS_SUBSAMPLE"] and len(df_ts) > CONFIG["TS_ROWS_SUBSAMPLE"]:
    df_ts = df_ts.sample(CONFIG["TS_ROWS_SUBSAMPLE"], random_state=SEED)

rename = {v: k for k, v in COLUMN_MAP.items() if v is not None}
df_ts = df_ts.rename(columns=rename)

# Parse date, sort
if "date" in df_ts.columns:
    df_ts["date"] = pd.to_datetime(df_ts["date"], errors="coerce")
    df_ts = df_ts.dropna(subset=["date"])

pollutant_cols = [c for c in ["pm25", "pm10", "no2", "so2", "co", "o3"] if c in df_ts.columns]
print("Pollutant columns present:", pollutant_cols)

# Numeric coercion + missing value handling
for c in pollutant_cols + ["aqi"]:
    df_ts[c] = pd.to_numeric(df_ts[c], errors="coerce")

group_key = "city" if "city" in df_ts.columns else None
if group_key:
    df_ts[pollutant_cols + ["aqi"]] = (
        df_ts.groupby(group_key)[pollutant_cols + ["aqi"]]
        .transform(lambda s: s.interpolate(limit_direction="both"))
    )
df_ts[pollutant_cols + ["aqi"]] = df_ts[pollutant_cols + ["aqi"]].fillna(
    df_ts[pollutant_cols + ["aqi"]].median()
)

# Outlier clipping (1st-99th percentile) — keeps extreme sensor errors from destroying scaling
for c in pollutant_cols + ["aqi"]:
    lo, hi = df_ts[c].quantile([0.01, 0.99])
    df_ts[c] = df_ts[c].clip(lo, hi)

df_ts = df_ts.dropna(subset=["aqi"]).reset_index(drop=True)
df_ts["aqi_category"] = df_ts["aqi"].apply(aqi_value_to_category)

print("Cleaned time-series shape:", df_ts.shape)
print(df_ts["aqi_category"].value_counts())
df_ts.head()


## 8. Feature engineering (rolling stats, lags, cyclical time features)

In [ ]:
def engineer_features(df, group_key, pollutant_cols):
    df = df.sort_values([group_key, "date"]) if group_key and "date" in df.columns else df.sort_index()
    g = df.groupby(group_key) if group_key else [(None, df)]

    parts = []
    iterator = g if group_key else [(None, df)]
    for _, sub in (iterator if group_key else [(None, df)]):
        sub = sub.copy()
        for c in pollutant_cols + ["aqi"]:
            sub[f"{c}_roll_mean_7"] = sub[c].rolling(7, min_periods=1).mean()
            sub[f"{c}_roll_std_7"]  = sub[c].rolling(7, min_periods=1).std().fillna(0)
            sub[f"{c}_lag_1"]       = sub[c].shift(1).fillna(sub[c])
            sub[f"{c}_lag_7"]       = sub[c].shift(7).fillna(sub[c])
            sub[f"{c}_diff_1"]      = sub[c].diff().fillna(0)
        parts.append(sub)
    out = pd.concat(parts, ignore_index=True) if group_key else parts[0]

    if "date" in out.columns:
        out["day_of_week"] = out["date"].dt.dayofweek
        out["month"] = out["date"].dt.month
        out["is_weekend"] = (out["day_of_week"] >= 5).astype(int)
        out["month_sin"] = np.sin(2 * np.pi * out["month"] / 12)
        out["month_cos"] = np.cos(2 * np.pi * out["month"] / 12)
        out["dow_sin"] = np.sin(2 * np.pi * out["day_of_week"] / 7)
        out["dow_cos"] = np.cos(2 * np.pi * out["day_of_week"] / 7)
    return out

df_ts = engineer_features(df_ts, group_key, pollutant_cols)

engineered_cols = [c for c in df_ts.columns if any(
    c.endswith(suf) for suf in ["_roll_mean_7", "_roll_std_7", "_lag_1", "_lag_7", "_diff_1"]
)] + [c for c in ["month_sin", "month_cos", "dow_sin", "dow_cos", "is_weekend"] if c in df_ts.columns]

TS_FEATURE_COLS = pollutant_cols + ["aqi"] + engineered_cols
print(f"Total engineered feature columns: {len(TS_FEATURE_COLS)}")
print(TS_FEATURE_COLS)


## 9. Build sequence windows for the BiLSTM branch

In [ ]:
SEQ_LEN = CONFIG["SEQ_LEN"]

def build_sequences(df, group_key, feature_cols, seq_len):
    X, y, cats = [], [], []
    groups = df.groupby(group_key) if group_key else [(None, df)]
    for _, sub in groups:
        sub = sub.sort_values("date") if "date" in sub.columns else sub
        feats = sub[feature_cols].values
        labels = sub["aqi_category"].values
        for i in range(len(sub) - seq_len):
            X.append(feats[i:i+seq_len])
            y.append(labels[i+seq_len])           # predict category AFTER the window
            cats.append(labels[i+seq_len])
        if len(X) > 60000:   # safety cap so this loop can't run away on huge datasets
            break
    return np.array(X, dtype=np.float32), np.array(y)

X_ts, y_ts_cat = build_sequences(df_ts, group_key, TS_FEATURE_COLS, SEQ_LEN)
print("Sequence tensor shape:", X_ts.shape, "| labels:", y_ts_cat.shape)

label_encoder = LabelEncoder()
label_encoder.fit(CONFIG["AQI_CATEGORIES"])
y_ts_int = label_encoder.transform(y_ts_cat)

scaler = StandardScaler()
n, t, f = X_ts.shape
X_ts_scaled = scaler.fit_transform(X_ts.reshape(-1, f)).reshape(n, t, f)
joblib.dump(scaler, os.path.join(CONFIG["OUTPUT_DIR"], "scaler.pkl"))


## 10. Load & label the image dataset (subsampled for time budget)

Images are resized and cached as a NumPy array in memory — at `IMG_SIZE=128` and
`IMG_SAMPLES_PER_CLASS≈700` this comfortably fits in RAM (~6 classes × 700 × 128×128×3 ≈ 400 MB).
If you set `FAST_MODE=False` with the full 12k images at 224px, switch to a `tf.data` streaming
pipeline instead (a commented-out version is included below) to avoid running out of RAM.


In [ ]:
IMG_SIZE = CONFIG["IMG_SIZE"]

def gather_image_paths():
    per_class = {}
    for folder in IMAGE_CLASSES:
        cat = FOLDER_TO_CATEGORY.get(folder)
        if cat is None:
            continue
        d = [c for c in class_dirs if os.path.basename(c) == folder][0]
        files = glob.glob(os.path.join(d, "**", "*.*"), recursive=True)
        files = [f for f in files if f.lower().endswith((".jpg", ".jpeg", ".png"))]
        random.shuffle(files)
        if CONFIG["IMG_SAMPLES_PER_CLASS"]:
            files = files[:CONFIG["IMG_SAMPLES_PER_CLASS"]]
        per_class[cat] = per_class.get(cat, []) + files
    return per_class

paths_by_cat = gather_image_paths()
for cat, files in paths_by_cat.items():
    print(cat, "->", len(files), "images")

def load_and_resize(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return img.numpy().astype(np.uint8)

img_paths, img_labels = [], []
for cat, files in paths_by_cat.items():
    for f in files:
        img_paths.append(f); img_labels.append(cat)

print(f"\nLoading {len(img_paths)} images into memory (this step can take a few minutes)...")
X_img = np.zeros((len(img_paths), IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
keep = []
for i, p in enumerate(tqdm(img_paths)):
    try:
        X_img[i] = load_and_resize(p)
        keep.append(i)
    except Exception:
        continue  # skip unreadable/corrupted files

X_img = X_img[keep]
y_img_cat = np.array(img_labels)[keep]
y_img_int = label_encoder.transform(y_img_cat)
print("Final image tensor:", X_img.shape)


## 11. Pair images with time-series windows (category-matched, as explained in the intro)

For each image, sample a time-series window from the **same AQI category**. This builds a joint
multimodal training set of matched `(image, sequence) -> category` triples.


In [ ]:
def pair_by_category(X_img, y_img_int, X_ts, y_ts_int, seed=SEED):
    rng = np.random.default_rng(seed)
    ts_by_class = {c: np.where(y_ts_int == c)[0] for c in np.unique(y_ts_int)}

    paired_img_idx, paired_ts_idx = [], []
    for i, cls in enumerate(y_img_int):
        pool = ts_by_class.get(cls)
        if pool is None or len(pool) == 0:
            continue
        j = rng.choice(pool)
        paired_img_idx.append(i)
        paired_ts_idx.append(j)
    return np.array(paired_img_idx), np.array(paired_ts_idx)

img_idx, ts_idx = pair_by_category(X_img, y_img_int, X_ts_scaled, y_ts_int)
print(f"Paired {len(img_idx)} multimodal samples "
      f"(out of {len(X_img)} images and {len(X_ts_scaled)} sequences).")

X_img_paired = X_img[img_idx]
X_ts_paired  = X_ts_scaled[ts_idx]
y_paired     = y_img_int[img_idx]   # same as y_ts_int[ts_idx] by construction

# Train / val / test split (stratified, no leakage across splits)
idx_all = np.arange(len(y_paired))
idx_train, idx_temp = train_test_split(idx_all, test_size=0.3, stratify=y_paired, random_state=SEED)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.5, stratify=y_paired[idx_temp], random_state=SEED)

def subset(idx):
    return X_img_paired[idx], X_ts_paired[idx], y_paired[idx]

Ximg_tr, Xts_tr, y_tr = subset(idx_train)
Ximg_val, Xts_val, y_val = subset(idx_val)
Ximg_test, Xts_test, y_test = subset(idx_test)
print("Train/Val/Test sizes:", len(y_tr), len(y_val), len(y_test))

n_classes = len(CONFIG["AQI_CATEGORIES"])
class_weights = compute_class_weight("balanced", classes=np.arange(n_classes), y=y_tr)
class_weight_dict = dict(enumerate(class_weights))
print("Class weights:", class_weight_dict)


## 12. tf.data pipelines (batching, augmentation, prefetch)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def augment_image(img):
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, 0.15)
    img = tf.image.random_contrast(img, 0.85, 1.15)
    img = tf.image.random_saturation(img, 0.85, 1.15)
    # Additional haze/fog augmentation to help CNN learn pollution cues
    img = tf.image.random_hue(img, 0.05)
    return img

MODAL_DROP_PROB = CONFIG["MODALITY_DROPOUT_PROB"]

def make_dataset(Ximg, Xts, y, training, batch_size):
    ds = tf.data.Dataset.from_tensor_slices(((Ximg, Xts), y))
    if training:
        ds = ds.shuffle(2048, seed=SEED)

    def _prep(pair, label):
        img, ts = pair
        img = tf.cast(img, tf.float32)
        if training:
            img = augment_image(img)
            # ── MODALITY DROPOUT ──────────────────────────────────────────────
            # ROOT CAUSE FIX: The model suffers from 'modal collapse' — it ignores
            # the image branch entirely because the timeseries labels (AQI values)
            # make it trivially easy to predict from timeseries alone.
            # Solution: Randomly zero out the timeseries input during training.
            # This forces the CNN branch to learn visual air quality cues (haze,
            # smoke, sky color, visibility) to maintain predictive accuracy.
            # ─────────────────────────────────────────────────────────────────
            if tf.random.uniform([]) < MODAL_DROP_PROB:
                ts = tf.zeros_like(ts)
        img = effnet_preprocess(img)
        label = tf.one_hot(label, n_classes)
        return (img, ts), label

    ds = ds.map(_prep, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(AUTOTUNE)
    return ds

train_ds = make_dataset(Ximg_tr, Xts_tr, y_tr, True, CONFIG["IMG_BATCH"])
val_ds   = make_dataset(Ximg_val, Xts_val, y_val, False, CONFIG["IMG_BATCH"])
test_ds  = make_dataset(Ximg_test, Xts_test, y_test, False, CONFIG["IMG_BATCH"])


## 13. Model — Image branch (EfficientNetB0, transfer learning)

In [ ]:
def build_image_branch(img_size):
    inp = layers.Input(shape=(img_size, img_size, 3), name="image_input")
    base = EfficientNetB0(include_top=False, weights="imagenet", input_tensor=inp)
    base.trainable = False  # phase 1: frozen backbone (fast)
    x = layers.GlobalAveragePooling2D()(base.output)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x)
    return inp, x, base


## 14. Model — Time-series branch (BiLSTM + attention pooling)

In [ ]:
class AttentionPool(layers.Layer):
    """Learns a weighted average over the time dimension instead of just taking the last hidden state."""
    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.score_dense = layers.Dense(units, activation="tanh")
        self.score_out = layers.Dense(1)

    def call(self, x):
        # x: (batch, time, features)
        scores = self.score_out(self.score_dense(x))            # (batch, time, 1)
        weights = tf.nn.softmax(scores, axis=1)                  # (batch, time, 1)
        context = tf.reduce_sum(weights * x, axis=1)             # (batch, features)
        return context, weights

def build_ts_branch(seq_len, n_features):
    inp = layers.Input(shape=(seq_len, n_features), name="ts_input")
    x = layers.Bidirectional(layers.LSTM(96, return_sequences=True, dropout=0.2))(inp)
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True, dropout=0.2))(x)
    context, attn_weights = AttentionPool(64, name="ts_attention_pool")(x)
    x = layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(context)
    return inp, x, attn_weights


## 15. Cross-attention fusion + classifier head

In [ ]:
class CrossAttentionFusion(layers.Layer):
    """Lightweight multi-head cross attention between the two 256-d branch embeddings."""
    def __init__(self, dim=256, num_heads=4, **kwargs):
        super().__init__(**kwargs)
        self.mha = layers.MultiHeadAttention(num_heads=num_heads, key_dim=dim // num_heads)
        self.norm = layers.LayerNormalization()

    def call(self, img_feat, ts_feat):
        img_seq = tf.expand_dims(img_feat, axis=1)   # (batch, 1, dim)
        ts_seq  = tf.expand_dims(ts_feat, axis=1)     # (batch, 1, dim)
        attn_out = self.mha(query=img_seq, key=ts_seq, value=ts_seq)
        fused = self.norm(attn_out[:, 0, :] + img_feat + ts_feat)
        return fused

def build_full_model(img_size, seq_len, n_ts_features, n_classes):
    img_inp, img_feat, effnet_base = build_image_branch(img_size)
    ts_inp, ts_feat, _ = build_ts_branch(seq_len, n_ts_features)

    fused = CrossAttentionFusion(dim=256, num_heads=4, name="cross_attention_fusion")(img_feat, ts_feat)
    x = layers.Dense(128, activation="relu")(fused)
    x = layers.Dropout(0.4)(x)
    out = layers.Dense(n_classes, activation="softmax", dtype="float32", name="aqi_category")(x)

    model = models.Model(inputs=[img_inp, ts_inp], outputs=out, name="multimodal_aqi_model")
    return model, effnet_base

model, effnet_base = build_full_model(CONFIG["IMG_SIZE"], SEQ_LEN, len(TS_FEATURE_COLS), n_classes)
model.summary()


## 16. Loss (Focal Loss) & training callbacks

In [ ]:
def categorical_focal_loss(gamma=2.0, label_smoothing=0.0):
    def loss_fn(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        if label_smoothing > 0:
            n = tf.cast(tf.shape(y_true)[-1], tf.float32)
            y_true = y_true * (1 - label_smoothing) + label_smoothing / n
        cross_entropy = -y_true * tf.math.log(y_pred)
        weight = tf.pow(1 - y_pred, gamma)
        return tf.reduce_sum(weight * cross_entropy, axis=-1)
    return loss_fn

lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=CONFIG["LEARNING_RATE"], decay_steps=1000
)
optimizer = tf.keras.optimizers.AdamW(learning_rate=lr_schedule, weight_decay=1e-5, clipnorm=1.0)

model.compile(
    optimizer=optimizer,
    loss=categorical_focal_loss(CONFIG["FOCAL_GAMMA"], CONFIG["LABEL_SMOOTHING"]),
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc", multi_label=True)],
)

ckpt_path = os.path.join(CONFIG["OUTPUT_DIR"], "best_model.keras")
cb_list = [
    callbacks.EarlyStopping(monitor="val_accuracy", patience=4, restore_best_weights=True),
    callbacks.ModelCheckpoint(ckpt_path, monitor="val_accuracy", save_best_only=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-6),
    callbacks.CSVLogger(os.path.join(CONFIG["OUTPUT_DIR"], "training_log.csv")),
]


## 17. Phase 1 training — frozen EfficientNet backbone (fast)

In [ ]:
t0 = time.time()
history_phase1 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=CONFIG["IMG_EPOCHS_HEAD"],
    class_weight=class_weight_dict,
    callbacks=cb_list,
    verbose=1,
)
print(f"Phase 1 done in {(time.time()-t0)/60:.1f} min")


## 18. Phase 2 fine-tuning — unfreeze the top layers of EfficientNet

In [ ]:
# FIXED: Unfreeze last 60 layers (was 30) for deeper visual adaptation
# With Modality Dropout forcing the CNN to predict on its own 35% of the time,
# we need to unfreeze more layers so it can actually adapt its representations.
effnet_base.trainable = True
for layer in effnet_base.layers[:-60]:  # was [:-30]
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=CONFIG["FINE_TUNE_LR"], weight_decay=1e-5, clipnorm=1.0),
    loss=categorical_focal_loss(CONFIG["FOCAL_GAMMA"], CONFIG["LABEL_SMOOTHING"]),
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc", multi_label=True)],
)

t0 = time.time()
history_phase2 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=CONFIG["IMG_EPOCHS_FINETUNE"],  # now 15 (was 5)
    class_weight=class_weight_dict,
    callbacks=cb_list,
    verbose=1,
)
print(f"Phase 2 (fine-tune) done in {(time.time()-t0)/60:.1f} min")

model.save(os.path.join(CONFIG["OUTPUT_DIR"], "final_model.keras"))


## 19. Training curves

In [ ]:
def plot_history(hist_list, keys=("accuracy", "loss", "auc")):
    fig, axes = plt.subplots(1, len(keys), figsize=(6 * len(keys), 4))
    for ax, k in zip(axes, keys):
        for h in hist_list:
            if k in h.history:
                ax.plot(h.history[k], label="train")
                ax.plot(h.history[f"val_{k}"], label="val")
        ax.set_title(k); ax.set_xlabel("epoch"); ax.legend()
    plt.tight_layout(); plt.show()

plot_history([history_phase1, history_phase2])


## 20. Full evaluation suite

In [ ]:
y_pred_probs = model.predict(test_ds)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = y_test

acc = accuracy_score(y_true, y_pred)
prec, rec, f1_w, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
_, _, f1_m, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
mcc = matthews_corrcoef(y_true, y_pred)
kappa = cohen_kappa_score(y_true, y_pred)

print(f"Accuracy          : {acc:.4f}")
print(f"Weighted F1        : {f1_w:.4f}")
print(f"Macro F1           : {f1_m:.4f}")
print(f"Matthews Corr Coef : {mcc:.4f}")
print(f"Cohen's Kappa      : {kappa:.4f}")

try:
    auc = roc_auc_score(tf.one_hot(y_true, n_classes).numpy(), y_pred_probs, multi_class="ovr")
    print(f"ROC-AUC (OvR)      : {auc:.4f}")
except Exception as e:
    print("ROC-AUC could not be computed:", e)

print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=CONFIG["AQI_CATEGORIES"], zero_division=0))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CONFIG["AQI_CATEGORIES"], yticklabels=CONFIG["AQI_CATEGORIES"])
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion Matrix")
plt.tight_layout(); plt.savefig(os.path.join(CONFIG["OUTPUT_DIR"], "confusion_matrix.png")); plt.show()


## 21. Explainability — Grad-CAM (image branch)

Grad-CAM is cheap (one backward pass per image), so it's kept as the primary visual explanation
method to stay inside the time budget. Score-CAM / Eigen-CAM / LIME are much slower (many forward
passes per image); if you have spare time after everything else finishes, the optional cell at
the end shows how to add Score-CAM.


In [ ]:
def find_last_conv_layer(m):
    for layer in reversed(m.layers):
        if isinstance(layer, layers.Conv2D) or "conv" in layer.name.lower() and len(layer.output_shape) == 4:
            return layer.name
    raise ValueError("No conv layer found")

last_conv_name = find_last_conv_layer(effnet_base)
print("Using layer for Grad-CAM:", last_conv_name)

grad_model = models.Model(
    [model.input], [effnet_base.get_layer(last_conv_name).output, model.output]
)

def grad_cam(img_array, ts_array, class_idx=None):
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model([img_array[None, ...], ts_array[None, ...]])
        if class_idx is None:
            class_idx = tf.argmax(preds[0])
        loss = preds[:, class_idx]
    grads = tape.gradient(loss, conv_out)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_out = conv_out[0]
    heatmap = tf.reduce_sum(conv_out * pooled_grads, axis=-1)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), int(class_idx)

n_examples = min(CONFIG["GRADCAM_EXAMPLES"], len(Ximg_test))
fig, axes = plt.subplots(2, n_examples, figsize=(3 * n_examples, 6))
for i in range(n_examples):
    raw_img = Ximg_test[i].astype(np.uint8)
    img_input = effnet_preprocess(raw_img.astype(np.float32))
    heatmap, pred_cls = grad_cam(img_input, Xts_test[i])
    heatmap_resized = tf.image.resize(heatmap[..., None], (IMG_SIZE, IMG_SIZE)).numpy().squeeze()

    axes[0, i].imshow(raw_img); axes[0, i].axis("off")
    axes[0, i].set_title(f"True: {CONFIG['AQI_CATEGORIES'][y_test[i]]}")
    axes[1, i].imshow(raw_img); axes[1, i].imshow(heatmap_resized, cmap="jet", alpha=0.5); axes[1, i].axis("off")
    axes[1, i].set_title(f"Pred: {CONFIG['AQI_CATEGORIES'][pred_cls]}")
plt.tight_layout(); plt.savefig(os.path.join(CONFIG["OUTPUT_DIR"], "gradcam_examples.png")); plt.show()


## 22. Explainability — SHAP (time-series/tabular branch)

We build a small surrogate model that maps the engineered tabular features to the predicted
class probabilities of the *fused* model (keeping the image fixed per sample), then run SHAP's
`KernelExplainer` on that surrogate with a small background set — this keeps SHAP fast while
still explaining which pollutant/engineered features drive the prediction.


In [ ]:
import shap

n_bg = min(CONFIG["SHAP_BACKGROUND_SAMPLES"], len(Xts_test))
n_explain = min(CONFIG["SHAP_EXPLAIN_SAMPLES"], len(Xts_test))

# Flatten the (seq_len, n_features) window to (seq_len*n_features,) for the tabular explainer,
# using the mean image embedding so we isolate the effect of the time-series features.
bg_idx = np.random.choice(len(Xts_test), n_bg, replace=False)
explain_idx = np.random.choice(len(Xts_test), n_explain, replace=False)

def ts_only_predict(flat_ts_batch):
    ts_batch = flat_ts_batch.reshape(-1, SEQ_LEN, len(TS_FEATURE_COLS))
    img_batch = np.repeat(Ximg_test[explain_idx[:1]].astype(np.float32), len(ts_batch), axis=0)
    img_batch = effnet_preprocess(img_batch)
    return model.predict([img_batch, ts_batch], verbose=0)

bg_flat = Xts_test[bg_idx].reshape(n_bg, -1)
explain_flat = Xts_test[explain_idx].reshape(n_explain, -1)

explainer = shap.KernelExplainer(ts_only_predict, bg_flat)
shap_values = explainer.shap_values(explain_flat, nsamples=100)

# Average feature importance across the time window, per raw feature name (not per timestep)
feature_names_per_step = TS_FEATURE_COLS
mean_abs_shap = np.mean(np.abs(np.array(shap_values)), axis=(0, 1))  # (seq_len*n_features,)
mean_abs_shap = mean_abs_shap.reshape(SEQ_LEN, len(TS_FEATURE_COLS)).mean(axis=0)

shap_df = pd.DataFrame({"feature": feature_names_per_step, "mean_abs_shap": mean_abs_shap})
shap_df = shap_df.sort_values("mean_abs_shap", ascending=False).head(20)

plt.figure(figsize=(8, 6))
sns.barplot(data=shap_df, x="mean_abs_shap", y="feature")
plt.title("SHAP feature importance — time-series branch (top 20)")
plt.tight_layout(); plt.savefig(os.path.join(CONFIG["OUTPUT_DIR"], "shap_importance.png")); plt.show()


## 23. Cross-attention weights (bonus interpretability — shows image vs. time-series contribution)

In [ ]:
attn_layer_model = models.Model(
    inputs=model.inputs,
    outputs=model.get_layer("cross_attention_fusion").output
)
# The MultiHeadAttention layer itself can return attention scores if called with
# return_attention_scores=True; for a quick global view we instead inspect the relative
# L2 norm of each branch's embedding vs. the fused output as a simple contribution proxy.
img_branch_model = models.Model(inputs=model.inputs[0], outputs=model.get_layer("dense").output) \
    if any(l.name == "dense" for l in model.layers) else None
print("Cross-attention fusion is active — see the layer 'cross_attention_fusion' in model.summary() above.")
print("For a per-sample attention heatmap, call the MultiHeadAttention layer directly with "
      "return_attention_scores=True on a rebuilt sub-model if you want this for your report figures.")


## 24. AQI category → health risk & recommendation module

In [ ]:
HEALTH_RISK_MAP = {
    "Good": {
        "risk": "Minimal", "asthma_risk": "Low", "outdoor_exercise": "Safe",
        "mask": "Not required", "children": "No restrictions",
    },
    "Satisfactory": {
        "risk": "Low", "asthma_risk": "Low-Moderate", "outdoor_exercise": "Generally safe",
        "mask": "Not required", "children": "No restrictions",
    },
    "Moderate": {
        "risk": "Moderate", "asthma_risk": "Moderate", "outdoor_exercise": "Reduce prolonged exertion",
        "mask": "Optional for sensitive groups", "children": "Limit prolonged outdoor play",
    },
    "Poor": {
        "risk": "High", "asthma_risk": "High", "outdoor_exercise": "Avoid prolonged/heavy exertion",
        "mask": "N95 recommended outdoors", "children": "Limit outdoor activity",
    },
    "Very Poor": {
        "risk": "Very High", "asthma_risk": "Very High", "outdoor_exercise": "Avoid outdoor exercise",
        "mask": "N95 strongly recommended", "children": "Stay indoors where possible",
    },
    "Severe": {
        "risk": "Severe / Emergency", "asthma_risk": "Extreme", "outdoor_exercise": "Avoid",
        "mask": "N95 mandatory outdoors", "children": "Stay indoors",
    },
}

def aqi_report(category: str) -> dict:
    info = HEALTH_RISK_MAP.get(category, HEALTH_RISK_MAP["Moderate"])
    return {"aqi_category": category, **info}

for cat in CONFIG["AQI_CATEGORIES"]:
    print(cat, "->", aqi_report(cat))


## 25. Save all artifacts for the FastAPI + NeonDB backend

These are the files your FastAPI service will load at startup: the trained model, the fitted
`StandardScaler`, the `LabelEncoder`, the feature-column order, and the health-risk mapping.


In [ ]:
out = CONFIG["OUTPUT_DIR"]
model.save(os.path.join(out, "aqi_multimodal_model.keras"))
joblib.dump(scaler, os.path.join(out, "scaler.pkl"))
joblib.dump(label_encoder, os.path.join(out, "label_encoder.pkl"))

with open(os.path.join(out, "class_mapping.json"), "w") as f:
    json.dump({
        "classes": CONFIG["AQI_CATEGORIES"],
        "aqi_breakpoints": CONFIG["AQI_BREAKPOINTS"],
        "ts_feature_columns": TS_FEATURE_COLS,
        "seq_len": SEQ_LEN,
        "img_size": IMG_SIZE,
        "health_risk_map": HEALTH_RISK_MAP,
    }, f, indent=2)

print("Saved artifacts in:", out)
print(os.listdir(out))


## 26. Inference demo (this is roughly what your FastAPI `/predict` endpoint will do)


In [ ]:
def predict_aqi(image_path: str, recent_pollutant_window: pd.DataFrame):
    """
    image_path: path to a single environmental photo
    recent_pollutant_window: DataFrame with the last SEQ_LEN rows, columns = TS_FEATURE_COLS
    """
    img = load_and_resize(image_path).astype(np.float32)
    img = effnet_preprocess(img)[None, ...]

    ts = recent_pollutant_window[TS_FEATURE_COLS].values.astype(np.float32)
    ts_scaled = scaler.transform(ts).reshape(1, SEQ_LEN, len(TS_FEATURE_COLS))

    probs = model.predict([img, ts_scaled], verbose=0)[0]
    pred_idx = int(np.argmax(probs))
    pred_cat = CONFIG["AQI_CATEGORIES"][pred_idx]

    return {
        "predicted_category": pred_cat,
        "confidence": float(probs[pred_idx]),
        "all_probabilities": {c: float(p) for c, p in zip(CONFIG["AQI_CATEGORIES"], probs)},
        "health_advisory": aqi_report(pred_cat),
    }

# Demo on one held-out test sample
demo_image_path = img_paths[keep[img_idx[idx_test[0]]]]
demo_window = pd.DataFrame(
    df_ts[TS_FEATURE_COLS].iloc[:SEQ_LEN].values, columns=TS_FEATURE_COLS
)
print(json.dumps(predict_aqi(demo_image_path, demo_window), indent=2))


## 27. Backend integration notes (FastAPI + NeonDB)

You mentioned you'll wire this up to a FastAPI backend with a NeonDB (Postgres) database — this
notebook's job stops at producing `aqi_multimodal_model.keras`, `scaler.pkl`, `label_encoder.pkl`
and `class_mapping.json` in the `aqi_model_artifacts/` folder above; download that whole folder
and load those four files at FastAPI startup, then call a function shaped like `predict_aqi()`
above inside your `/predict` route. A reasonable NeonDB schema to start with:

- `users(id, email, ...)`
- `predictions(id, user_id, image_path, predicted_category, confidence, created_at)`
- `pollution_readings(id, city, date, pm25, pm10, no2, so2, co, o3, aqi)`
- `health_risk(category, risk_level, advisory_text)`

That's outside the scope of this notebook (it's a separate backend project), but the artifact
files above are exactly what that FastAPI service needs to load.


## 28. Optional extensions (only if you have spare time after the 3–4 hr run)

Uncomment/adapt these if you want to push accuracy or explainability further and have a stronger
GPU or more time budget available:
- Set `CONFIG["FAST_MODE"] = False` to use the full image set at 224px with more epochs
- Add Score-CAM / Eigen-CAM (slower, many forward passes per image, but gradient-free)
- Add LIME on the image branch for a second, independent explanation method for your viva
- Try `ConvNeXtTiny` instead of `EfficientNetB0` for the image branch (larger, slower, usually
  a bit more accurate)
- Replace category-matched pairing with a true city/date join if you find the image dataset has
  usable location metadata after inspecting the printout from Cell 4
